# Примеры утечки данных

## Утечка обучающего/тестового множества

**Суть эксперимента**: создаётся задача бинарной классификации с 10 000 случайных признаков и 200 объектами. Поскольку признаки и целевая переменная сгенерированы случайно, они независимы. Значит, честная точность любой модели должна быть около 0.5 (случайное угадывание).

In [1]:
import numpy as np
n_samples, n_features, n_classes = 200, 10000, 2
rng = np.random.RandomState(42)
X = rng.standard_normal((n_samples, n_features))
y = rng.choice(n_classes, n_samples)

**Неправильно**

Сначала на всех данных выполняется отбор 25 лучших признаков. Потом данные делятся на train/test. Результат обучения модели гораздо более высокий, чем ожидалось:

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

X_selected = SelectKBest(k=25).fit_transform(X, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, random_state=42)
gbc = GradientBoostingClassifier(random_state=1)
gbc.fit(X_train, y_train)
GradientBoostingClassifier(random_state=1)

y_pred = gbc.predict(X_test)
accuracy_score(y_test, y_pred)

0.76

**Почему это произошло?**

Метод выбора лучших признаков "подсмотрел" тестовые данные. Он выбрал те признаки, которые случайно коррелируют с целевой переменной именно в этой тестовой выборке. Модель поняла на что нужно обратить внимание.

**Правильно**

Чтобы предотвратить утечку данных, рекомендуется сначала разделить данные на обучающие и тестовые подмножества, а только потом отбор признаков только на train. К тесту применяется уже обученный трансформер. Обучаем модель и получаем точность ближе к 0.5.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42)
select = SelectKBest(k=25)
X_train_selected = select.fit_transform(X_train, y_train)

gbc = GradientBoostingClassifier(random_state=1)
gbc.fit(X_train_selected, y_train)
GradientBoostingClassifier(random_state=1)

X_test_selected = select.transform(X_test)
y_pred = gbc.predict(X_test_selected)
accuracy_score(y_test, y_pred)

0.46

Рекомендуют использовать Pipeline, чтобы воедино связать выбор функции и оценки модели. Гарантируется, что при запуске используются только обучающие данные, а тестовые используются только для вычисления точности.

In [4]:
from sklearn.pipeline import make_pipeline
X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42)
pipeline = make_pipeline(SelectKBest(k=25),
                        GradientBoostingClassifier(random_state=1))
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
accuracy_score(y_test, y_pred)

0.46

Кросс валидация гарантирует, что во время подбора и прогнозирования используются правильные подмножества данных и метод оценки.

In [5]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(pipeline, X, y)

round(scores.mean(), 3)

np.float64(0.46)

## Утечка целевой переменной

**Суть эксперимента**: создается задача бинарной классификации на 1000 объектов. Признаки (`age`, `tenure`, `n_products`) сгенерированы случайно и независимо от таргета. Дополнительно в данные попадает признак `refund_after_churn` - сумма возврата (она присутсвует только у ушедших клиентов). Честная точность модели должна быть около 0.5.

In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [7]:
rng = np.random.RandomState(42)
n_samples = 1000

X_honest = pd.DataFrame({
    "age":        rng.normal(40, 12, n_samples),
    "tenure":     rng.exponential(24, n_samples),
    "n_products": rng.normal(1, 5, n_samples),
})
y = rng.choice([0, 1], n_samples)

leak = np.where(y == 1, rng.normal(100, 10, n_samples), 0.0)
X_leaked = X_honest.copy()
X_leaked["refund_after_churn"] = leak

**Неправильно**

Признак-утечка остаётся в данных, модель обучается на нём вместе с остальными.

In [8]:
X_tr, X_te, y_tr, y_te = train_test_split(X_leaked, y, random_state=42)
model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
round(accuracy_score(y_te, model.predict(X_te)), 3)

1.0

**Почему так произошло?**

Признак `refund_after_churn` есть только у ушедших клиентов. Модель выучила правило, что если refund > 0, то churn = 1. Формально это не копия таргета, но содержится информация о нем.

**Правильно**

Признак-утечку надо удалить из данных до обучения.

In [9]:
X_tr, X_te, y_tr, y_te = train_test_split(X_honest, y, random_state=42)
model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
round(accuracy_score(y_te, model.predict(X_te)), 3)

0.464

Рекомендуется перед обучением проверять каждый признак на "доступность во время предсказания". Pipeline не поможет, т.к. утечка находится в саммих данных, а не в порядке обработки.

## Утечка на уровне признаков

**Суть эксперимента:** создается задача бинарной классификации на 1000 объектов. Таргет сгенерирован так, что по началу идут объекты с `y = 0`, потом `y = 1`. Признаки `age`, `income`, `city`, `n_products` не зависят от таргета. В данных есть два "слишком хороших" признака:

1. `row_id` – монотонный идентификатор, коррелирует с таргетом.
2. `city_te_leaked` – target encoding, посчитанный на **всём** датасете до split.

Точность модели должна быть около 0.5.

In [10]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [11]:
rng = np.random.RandomState(42)
n_samples = 1000

y = np.array([0] * (n_samples // 2) + [1] * (n_samples // 2))

n_cities = 100
cities = [f"City_{i}" for i in range(n_cities)]
city_col = rng.choice(cities, n_samples)

X_honest = pd.DataFrame({
    "age":        rng.normal(40, 12, n_samples),
    "income":     rng.normal(50_000, 15_000, n_samples),
    "city":       city_col,
    "n_products": rng.randint(1, 5, n_samples),
})

row_id = np.arange(n_samples)

city_target_mean = pd.Series(y).groupby(X_honest["city"]).mean()
X_honest["city_te_leaked"] = X_honest["city"].map(city_target_mean)

X_leaked = X_honest.copy()
X_leaked["row_id"] = row_id
X_leaked = X_leaked.drop(columns=["city"])

**Неправильно**

`row_id`, `city_te_leaked` остаются в данных. Модель обучается на них.

In [12]:
from sklearn.ensemble import RandomForestClassifier

X_tr, X_te, y_tr, y_te = train_test_split(X_leaked, y, random_state=42)
model = RandomForestClassifier(random_state=1, max_depth=10).fit(X_tr, y_tr)
round(accuracy_score(y_te, model.predict(X_te)), 3)

1.0

**Почему это произошло?**

- `row_id` монотонно растёт, а таргет сгенерирован в порядке возрастания: первая половина ID соответствует y = 0, вторая y = 1. Дерево просто режет по порогу ID.

- `city_te_leaked` посчитан на всём датасете, включая тестовые объекты. В каждом городе всего ~10 записей, поэтому среднее по таргету в городе случайно близко к 0 или 1 модель выучила эти «шпаргалки» и на train, и на test.

**Правильно**

ID - удаляется. Target encoding считается только на train.

In [13]:
X_honest = X_honest.drop(columns=["city_te_leaked"])

X_tr, X_te, y_tr, y_te = train_test_split(X_honest, y, random_state=42)

y_tr_s = pd.Series(y_tr, index=X_tr.index)
te_map = y_tr_s.groupby(X_tr["city"]).mean()

X_tr_te = X_tr.assign(city_te=X_tr["city"].map(te_map).fillna(y_tr.mean()))
X_te_te = X_te.assign(city_te=X_te["city"].map(te_map).fillna(y_tr.mean()))

model = LogisticRegression(max_iter=1000).fit(
    X_tr_te.drop(columns=["city"]), y_tr)

round(accuracy_score(y_te, model.predict(X_te_te.drop(columns=["city"]))), 3)

0.528

Рекомендуется ипользовать Pipeline с TargetEncoder - он гарантирует, что кодирование обучается только на train.

In [14]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import TargetEncoder
from sklearn.compose import ColumnTransformer

pre = ColumnTransformer([
    ("te", TargetEncoder(random_state=42), ["city"]),
], remainder="passthrough")

pipe = make_pipeline(pre, LogisticRegression(max_iter=1000))

X_tr, X_te, y_tr, y_te = train_test_split(X_honest, y, random_state=42)
pipe.fit(X_tr, y_tr)

round(accuracy_score(y_te, pipe.predict(X_te)), 3)

0.56